# KisaanDost - Varhadi ASR adaptation (Experiment B) on Colab GPU

Fine-tunes `ai4bharat/indicconformer_stt_mr_hybrid_ctc_rnnt_large` on the Varhadi data bundle built locally by
`experiments/varhadi_adaptation/build_manifests.py`, using the AI4Bharat NeMo fork's own
`examples/asr/speech_to_text_finetune.py` and `experiments/varhadi_adaptation/finetune_varhadi.yaml`.

**Before running:** read `P2_READY.md`. Do not train until a verified test set exists and you have decided whether
to train on unverified aligned candidates (`bundle_info.json` records that choice).

What you need:
1. Colab runtime with a GPU (Runtime > Change runtime type > T4 or better).
2. A Colab secret named `HF_TOKEN` (key icon in the left bar) for a HuggingFace account that accepted the model's terms.
   The token is never written into this notebook.
3. `kisaandost_p2/p2_bundle.zip` in your Google Drive (manifests + 16 kHz chapter WAVs; transcripts stay private on Drive).
4. The KisaanDost code pushed to GitHub (or upload a zip and adjust the clone cell).

Scoring is NOT done here: download the best `.nemo` and run `scripts/run_gold_eval.py --checkpoint` locally, so the
adapted model is scored by exactly the same pipeline, test set and normalisation as the baseline.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
P2 = '/content/drive/MyDrive/kisaandost_p2'
import os; os.makedirs(P2, exist_ok=True); print(os.listdir(P2))

## 1. Environment: AI4Bharat NeMo fork at the commit used locally, plus the pins that local setup needed

In [ ]:
%%bash
set -e
cd /content
[ -d NeMo ] || git clone https://github.com/AI4Bharat/NeMo.git
cd NeMo
git checkout 8dce88cf8e94963e2033c3137f7b9993b51db88a   # nemo-v2, same commit as the local baseline env
bash reinstall.sh
# pins found necessary locally (project_progress.txt section 3)
pip install "pytorch-lightning==2.2.1" "datasets>=2.20,<3" "jiwer>=3.0,<5"

In [ ]:
# NeMo 1.23's audio loader reads np.sctypes, removed in NumPy 2 (same shim as asr_baseline/model.py)
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {'int': [np.int8, np.int16, np.int32, np.int64], 'float': [np.float16, np.float32, np.float64]}
import torch, nemo
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| nemo', nemo.__version__)

## 2. Project code and data bundle

In [ ]:
%%bash
cd /content
[ -d KisaanDost ] || git clone https://github.com/atharvaGujarathi0405/Varhadi_Bible_Audio.git KisaanDost
cd KisaanDost && git log -1 --oneline

In [ ]:
%%bash
cd /content
unzip -q -o /content/drive/MyDrive/kisaandost_p2/p2_bundle.zip -d /content/
cat /content/p2_bundle/bundle_info.json

In [ ]:
# sanity checks: every manifest line has lang=mr, offset/duration, and its audio file exists
import json, os
root = '/content/p2_bundle'
for split in ['train', 'val', 'test']:
    path = f'{root}/manifests/{split}_manifest.json'
    lines = [json.loads(l) for l in open(path, encoding='utf-8')]
    missing = {l['audio_filepath'] for l in lines if not os.path.exists(os.path.join(root, l['audio_filepath']))}
    assert all(l['lang'] == 'mr' for l in lines) and not missing, (split, missing)
    hours = sum(l['duration'] for l in lines) / 3600
    labels = {s: sum(l['label_source'] == s for l in lines) for s in {l['label_source'] for l in lines}}
    print(f'{split}: {len(lines)} segments, {hours:.2f} h, labels {labels}')

## 3. Base checkpoint (gated repo; token from Colab secrets)

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
base = hf_hub_download(repo_id='ai4bharat/indicconformer_stt_mr_hybrid_ctc_rnnt_large',
                       filename='indicconformer_stt_mr_hybrid_rnnt_large.nemo',   # see asr_baseline/config.py
                       token=userdata.get('HF_TOKEN'))
open('/content/base_nemo_path.txt', 'w').write(base)
print(base)

## 4. Fine-tune

Manifests use paths relative to the bundle, so training runs from `/content/p2_bundle`.
Checkpoints go to Drive (`exp_manager.exp_dir`); `resume_if_exists=true` continues after a disconnect.
Optional first check: add `trainer.max_steps=20` to confirm memory/throughput, note the step time in `P2_READY.md`, then run fully.

In [ ]:
%%bash
cd /content/p2_bundle
python /content/NeMo/examples/asr/speech_to_text_finetune.py \
  --config-path=/content/KisaanDost/experiments/varhadi_adaptation \
  --config-name=finetune_varhadi \
  init_from_nemo_model=$(cat /content/base_nemo_path.txt) \
  model.train_ds.manifest_filepath=/content/p2_bundle/manifests/train_manifest.json \
  model.validation_ds.manifest_filepath=/content/p2_bundle/manifests/val_manifest.json \
  2>&1 | tee -a /content/drive/MyDrive/kisaandost_p2/train_log.txt

## 5. Collect outputs

In [ ]:
%%bash
find /content/drive/MyDrive/kisaandost_p2/experiments -name "*.nemo" -printf "%T+ %s %p\n" | sort

Download the best `.nemo` (lowest `val_wer` in its filename) and, locally in WSL:

```bash
python scripts/run_gold_eval.py --name varhadi_adapted --checkpoint /path/to/best.nemo --test-chapters JHN_001
python experiments/evaluation/compare.py
```

Record in `EXPERIMENT_LOG.md`: bundle_info.json, this notebook's config overrides, GPU type, step time, epochs run,
best val_wer and the local test-set scores. Never call the adapted model "better" unless the measured test scores say so.